# Hlavný experiment: pôvodné inštancie autorov (Colab)

Spustí **našu implementáciu** aj **pôvodný C++ program autorov** na pôvodných inštanciách
Henneberg & Neufeld (2016), ktoré autori zverejnili 11. 11. 2026, a výsledky ukladá priebežne
na **Google Drive** (prerušený beh sa po opätovnom spustení dokončí, hotové behy sa preskočia).

Postup: v bunke nižšie nastav `SESSION` → **Runtime → Run all** → povoľ prístup k Drive.
Rozdelenie na session a odhad času: `colab/README.md` v repozitári.

In [ ]:
# ======== NASTAVENIA ========
SESSION = 1                # 1–4, každá session iné číslo (tabuľka v colab/README.md)
BRANCH = "main"
DRIVE_DIR = "/content/drive/MyDrive/npfs-mo-povodne"
# ============================
SESSIONS = {
    # n od–do, seedy, naša implementácia, program autorov, naša s výberom jobov ako autori
    1: dict(n_min=5,   n_max=50,  seeds=list(range(1, 11)), ours=True,  authors=True,  sampling=False),
    2: dict(n_min=100, n_max=100, seeds=list(range(1, 6)),  ours=True,  authors=True,  sampling=False),
    3: dict(n_min=100, n_max=100, seeds=list(range(6, 11)), ours=True,  authors=True,  sampling=False),
    4: dict(n_min=30,  n_max=100, seeds=list(range(1, 11)), ours=False, authors=False, sampling=True),
}
CFG = SESSIONS[SESSION]
TAG = f"session{SESSION}"
print(TAG, CFG)

In [ ]:
# 1) Google Drive – sem sa zapisujú výsledky (prežijú odpojenie runtime)
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Výsledky:", DRIVE_DIR)

In [ ]:
%%bash -s "$BRANCH"
# 2) Náš repozitár, Python balíky a pôvodné inštancie (src/fetch_original.py ich stiahne
#    z repozitára autorov na pevnom commite a skontroluje voči ich table2.csv)
set -e
cd /content
rm -rf replication-npfs-mo
git clone -q --depth 1 -b "$1" https://github.com/svihuralukas/replication-npfs-mo.git
if [ ! -f replication-npfs-mo/src/fetch_original.py ]; then
    # PR s pôvodnými inštanciami ešte nie je zlúčený do main -> vetva PR
    echo "Vetva $1 ešte neobsahuje pôvodné inštancie, používam etapa-3-povodne-instancie"
    rm -rf replication-npfs-mo
    git clone -q --depth 1 -b etapa-3-povodne-instancie https://github.com/svihuralukas/replication-npfs-mo.git
fi
cd replication-npfs-mo
git log --oneline -1
pip -q install -r requirements.txt
python --version
python src/fetch_original.py

In [ ]:
%%bash
# 3) Pôvodný program autorov – rovnaký commit ako v našich referenčných behoch (kód sa odvtedy nezmenil,
#    pribudli len inštancie). CMake projektu chce Boost >= 1.75, Colab má 1.74, preto priamo cez g++
#    (overené: výsledky sú bit po bite rovnaké ako z buildu cez CMake).
set -e
apt-get -qq update > /dev/null
DEBIAN_FRONTEND=noninteractive apt-get -qq install -y libboost-program-options-dev libboost-timer-dev \
    libboost-system-dev libboost-chrono-dev libboost-filesystem-dev > /dev/null
cd /content
rm -rf npfs-mo
git clone -q https://github.com/mrpritt/npfs-mo.git
cd npfs-mo
git checkout -q bde1bc3
git submodule -q update --init
cd src
g++ -O3 -DNDEBUG -std=c++17 -DIL_STD -fexceptions -Ifmt/include \
    npfsmo.cpp instance.cpp solution.cpp heuristics.cpp holes.cpp options.cpp random.cpp logging.cpp helpers.cpp \
    -o /content/npfsmo -lboost_program_options -lboost_timer -lboost_system -lboost_chrono -lboost_filesystem -lrt -ldl
/content/npfsmo --flowtime --timelimit -1 --iterfactor 0.1 --npfs /content/replication-npfs-mo/instances/original/0.2_05_05_00.txt | grep INFO
echo "(posledné ΣC v riadku INFO musí byť 1191 – optimum z Obr. 3 článku)"

In [ ]:
# 4) Kontrola prostredia (~3 min):
#    - testy 2. fázy, NPS-set a IGA,
#    - program autorov so seedom 1 musí dať presne hodnoty z článku (data/authors, po inštanciách),
#    - naša implementácia so seedom 2 musí dať presne to isté ako na notebooku.
import os, subprocess, csv
os.chdir("/content/replication-npfs-mo")
r = subprocess.run(["python", "-m", "pytest", "tests/test_nonperm.py", "tests/test_nps_set.py",
                    "tests/test_iga.py", "-q"], capture_output=True, text=True)
print(r.stdout.strip().splitlines()[-1])
assert r.returncode == 0, r.stdout

CHECK = ["0.3_20_10_00", "0.4_50_20_01"]
files = [f"instances/original/{i}.txt" for i in CHECK]
tmp = "/content/check"
os.makedirs(tmp, exist_ok=True)
paper = {}
for f, col, key in (("table3-iga.csv", "igapols.ft", "iga_ft"), ("table6.csv", "igapo.ft", "iga0_ft"),
                    ("table6.csv", "igapo.nps.ft", "nps_ft")):
    for row in csv.DictReader(open(f"data/authors/{f}")):
        name = f"{float(row['d']):.1f}_{int(row['n']):02d}_{int(row['m']):02d}_{int(row['i']):02d}"
        paper.setdefault(name, {})[key] = row[col]
# naša implementácia, seed 2, hodnoty z notebooku (iga0_ft, nps_ft, iga_ft)
expected = {"uniform": {"0.3_20_10_00": ("14160", "14160", "14091"), "0.4_50_20_01": ("74780", "74140", "73345")},
            "autori":  {"0.3_20_10_00": ("14160", "14160", "14091"), "0.4_50_20_01": ("75476", "74750", "74217")}}
ok = True
subprocess.run(["python", "reference/run_authors.py", "--npfsmo", "/content/npfsmo", "--instances", *files,
                "--seeds", "1", "--out", f"{tmp}/a.csv"], check=True, capture_output=True)
for row in csv.DictReader(open(f"{tmp}/a.csv")):
    same = all(row[k] == paper[row["instance"]][k] for k in ("iga0_ft", "nps_ft", "iga_ft"))
    ok &= same
    print(f"autori, seed 1  {row['instance']}: IGA ΣC {row['iga_ft']} vs článok {paper[row['instance']]['iga_ft']}",
          "OK" if same else "NEZHODA")
for sampling, exp in expected.items():
    out = f"{tmp}/o_{sampling}.csv"
    subprocess.run(["python", "src/run_experiments.py", "--instances", *files, "--seeds", "2",
                    "--sampling", sampling, "--out", out], check=True, capture_output=True)
    for row in csv.DictReader(open(out)):
        same = (row["iga0_ft"], row["nps_ft"], row["iga_ft"]) == exp[row["instance"]]
        ok &= same
        print(f"naša ({sampling:7s}) {row['instance']}: IGA ΣC {row['iga_ft']} vs {exp[row['instance']][2]}",
              "OK" if same else "NEZHODA")
assert ok, "Výsledky sa nezhodujú s referenciou – NEPOKRAČOVAŤ, napíš mi."
print("Prostredie je v poriadku.")

In [ ]:
# 5) Hlavný beh: pre každý seed naša implementácia, potom program autorov (podľa nastavení session).
#    Zapisuje sa priebežne na Drive; po odpojení stačí znova Run all (hotové sa preskočí).
import os, subprocess, platform, datetime
os.chdir("/content/replication-npfs-mo")
jobs = os.cpu_count()
cpu = subprocess.run("lscpu | grep 'Model name' | cut -d: -f2", shell=True, capture_output=True, text=True).stdout.strip()
with open(f"{DRIVE_DIR}/info_{TAG}.txt", "a") as f:
    f.write(f"{datetime.datetime.now().isoformat()} start {TAG} {CFG} cpu='{cpu}' cores={jobs} "
            f"python={platform.python_version()}\n")
print("CPU:", cpu, "| jadrá:", jobs)

inst = ["--instances", "instances/original/*.txt", "--min-n", str(CFG["n_min"]), "--max-n", str(CFG["n_max"]),
        "--jobs", str(jobs)]
for s in CFG["seeds"]:
    if CFG["ours"]:
        print(f"\n=== seed {s}: naša implementácia ===", flush=True)
        subprocess.run(["python", "src/run_experiments.py", *inst, "--seeds", str(s),
                        "--out", f"{DRIVE_DIR}/iga_ours_{TAG}.csv"], check=True)
    if CFG["authors"]:
        print(f"\n=== seed {s}: program autorov ===", flush=True)
        subprocess.run(["python", "reference/run_authors.py", "--npfsmo", "/content/npfsmo", *inst,
                        "--seeds", str(s), "--out", f"{DRIVE_DIR}/iga_authors_{TAG}.csv"], check=True)
    if CFG["sampling"]:
        print(f"\n=== seed {s}: naša implementácia, výber jobov ako autori ===", flush=True)
        subprocess.run(["python", "src/run_experiments.py", *inst, "--seeds", str(s), "--sampling", "autori",
                        "--out", f"{DRIVE_DIR}/iga_ours-sampling-autori_{TAG}.csv"], check=True)
with open(f"{DRIVE_DIR}/info_{TAG}.txt", "a") as f:
    f.write(f"{datetime.datetime.now().isoformat()} done {TAG}\n")
print("\nHOTOVO")

In [ ]:
# 6) Súhrn – počet hotových behov na seed v každom súbore
#    (cieľ: session 1 → 1800 na seed, session 2 a 3 → 360, session 4 → 1080)
import pandas as pd, os
for f in sorted(os.listdir(DRIVE_DIR)):
    if f.endswith(".csv") and "history" not in f:
        d = pd.read_csv(f"{DRIVE_DIR}/{f}")
        print(f, d.groupby("seed").size().to_dict())